In [2]:
from langchain_anyllm import ChatAnyLLM

llm = ChatAnyLLM(
        # model="qwen/qwen3.5-9b",   # must match the model loaded in LMStudio
        # model="qwen3.5-4b",   # must match the model loaded in LMStudio
        provider='lmstudio',
        model="google/gemma-4-e4b",
        api_base="http://localhost:1234/v1",
        api_key="not-needed",
    )

In [3]:
llm.invoke('who are you')

AIMessage(content='I am Gemma 4, an open weights Large Language Model developed by Google DeepMind.\n\nI am designed to process information, answer questions, generate text, and assist with a wide range of tasks by understanding and generating human language.', additional_kwargs={}, response_metadata={'token_usage': CompletionUsage(completion_tokens=334, prompt_tokens=19, total_tokens=353, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=283, rejected_prediction_tokens=None), prompt_tokens_details=None), 'model': 'google/gemma-4-e4b', 'finish_reason': 'stop', 'model_name': 'google/gemma-4-e4b'}, id='lc_run--019eabc6-0cff-7fb3-b28b-3dd6fd6c5194-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 334, 'total_tokens': 353})

In [4]:
from pprint import pprint

def inspect_model(model):
    print("\n=== MODEL OBJECT ===")
    print(type(model))

    print("\n=== DIRECT ATTRIBUTES ===")
    for name in [
        "model",
        "model_name",
        "provider",
        "profile",
        "max_tokens",
        "model_kwargs",
    ]:
        print(f"{name}: {getattr(model, name, None)}")

    profile = getattr(model, "profile", None)

    print("\n=== PROFILE ===")
    pprint(profile)

    print("\n=== max_input_tokens ===")
    if isinstance(profile, dict):
        print(profile.get("max_input_tokens"))
    else:
        print(None)

    print("\n=== DIR CONTAINS PROFILE-ish FIELDS ===")
    for name in dir(model):
        lowered = name.lower()
        if "profile" in lowered or "token" in lowered or "context" in lowered:
            print(name)

In [5]:
inspect_model(llm)


=== MODEL OBJECT ===
<class 'langchain_anyllm.chat_models.ChatAnyLLM'>

=== DIRECT ATTRIBUTES ===
model: google/gemma-4-e4b
model_name: None
provider: lmstudio
profile: None
max_tokens: None
model_kwargs: {}

=== PROFILE ===
None

=== max_input_tokens ===
None

=== DIR CONTAINS PROFILE-ish FIELDS ===
_check_profile_keys
_resolve_model_profile
_set_model_profile
custom_get_token_ids
get_num_tokens
get_num_tokens_from_messages
get_token_ids
max_tokens
profile


In [ ]:
# from pprint import pprint
# from langchain.chat_models import init_chat_model


# def inspect_model(model):
#     print("\n=== MODEL OBJECT ===")
#     print(type(model))

#     profile = getattr(model, "profile", None)

#     print("\n=== PROFILE ===")
#     pprint(profile)

#     print("\n=== max_input_tokens ===")
#     if isinstance(profile, dict):
#         print(profile.get("max_input_tokens"))
#     else:
#         print(None)


# def main():
#     model = init_chat_model(
#         "openai:gpt-4.1",  # replace with your AnyLLM-style init string if supported
#         temperature=0,
#     )

#     inspect_model(model)


# if __name__ == "__main__":
#     main()

In [2]:
from deepagents import create_deep_agent, FilesystemPermission
from deepagents.backends import FilesystemBackend
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver

root = r'D:\Projects\mira'


# # ---------------------------------------------------------------------------
# # Human-in-the-loop: which tools need approval and what decisions are allowed
# # deepagents pauses execution, we collect decisions, then resume with Command
# # ---------------------------------------------------------------------------
# INTERRUPT_ON = {
#     # Filesystem reads — safe, no approval needed
#     "read_file": False,
#     "ls":        False,
#     "glob":      False,
#     "grep":      False,
#     # Writes — show diff, let user approve/edit/reject
#     "write_file": {"allowed_decisions": ["approve", "edit", "reject"]},
#     "edit_file":  {"allowed_decisions": ["approve", "edit", "reject"]},
#     # eval (interpreter) — approve/reject; no edit since it's a code block
#     "eval":       {"allowed_decisions": ["approve", "reject"]},
# }

# # ---------------------------------------------------------------------------
# # PTC allowlist: which tools the interpreter can call from TypeScript code.
# # e.g. the agent can write:
# #   const files = await tools.glob({ pattern: "**/*.py" });
# #   const contents = await Promise.all(files.map(f => tools.readFile({ path: f })));
# # ---------------------------------------------------------------------------
# PTC_TOOLS = ["read_file", "write_file", "edit_file", "ls", "glob", "grep"]


# # FilesystemBackend provides ls/read_file/write_file/edit_file/glob/grep.
# # virtual_mode=True enforces root_dir-relative path semantics.
backend = FilesystemBackend(root_dir=root, virtual_mode=True)

# # First-match-wins permission rules.
# permissions = [
#     # Never expose secrets
#     FilesystemPermission(
#         operations=["read", "write"],
#         paths=["/**/.env", "/**/.env.*", "/**/*.pem", "/**/*.key"],
#         mode="deny",
#     ),
#     # Allow everything else inside the workspace
#     FilesystemPermission(
#         operations=["read", "write"],
#         paths=["/**"],
#         mode="allow",
#     ),
# ]

# # CodeInterpreterMiddleware adds the `eval` tool.
# # ptc= allowlists which agent tools are callable from inside TypeScript
# # as async functions under the global `tools` namespace.
# interpreter = CodeInterpreterMiddleware(ptc=PTC_TOOLS)

# system_prompt = (
#     "You are Mira (Minimal Iterative Reasoning Agent), a helpful coding assistant running in the user's terminal.\n"
#     "You have filesystem tools (read_file, write_file, edit_file, ls, glob, grep) "
#     "and an eval tool that runs TypeScript in a persistent QuickJS interpreter.\n"
#     "Use the interpreter (eval) when you need to: loop over many files, run "
#     "parallel tool calls with Promise.all, filter/aggregate data, or keep "
#     "intermediate state out of the model context.\n"
#     "Always explain what you are about to do before doing it. "
#     "Be concise. Prefer showing code over describing it. "
#     "If a task is ambiguous, ask one clarifying question before proceeding."
# )

# agent = create_deep_agent(
#     model=llm,
#     backend=backend,
#     permissions=permissions,
#     middleware=[interpreter],
#     interrupt_on=INTERRUPT_ON,
#     # checkpointer=MemorySaver(),
#     system_prompt=system_prompt,
# )

In [7]:
agent = create_deep_agent(
    model=llm,
    backend=backend,
    # permissions=permissions,
    # middleware=[interpreter],
    # interrupt_on=INTERRUPT_ON,
    # checkpointer=MemorySaver(),
    # system_prompt=system_prompt,
)

In [8]:
user_input = 'what files are in the root folder'
agent_input = {"messages": [{"role": "user", "content": user_input}]}

In [9]:
# run = agent.stream_events(
#     agent_input, 
#     config={"configurable": {"thread_id": 'xxxxx'}}, 
#     version="v3"
#     )


# for message in run.messages:
#     header_printed = False

#     for delta in message.reasoning:
#         if delta:
#             # print_reasoning_header()
#             # print_reasoning_token(delta)
#             # print(delta)
#             pass

#     for delta in message.text:
#         if delta:
#             if not header_printed:
#                 # print_agent_header()
#                 header_printed = True
#             # print(delta)

#     if header_printed:
#         # print_newline()
#         # print()
#         pass

# # --- Tool calls: finalized name + args + output ---
# # run.tool_calls is a separate projection that does NOT share state with
# # run.messages, so iterating it after messages is safe.
# for call in run.tool_calls:
#     args = call.input if isinstance(call.input, dict) else {"input": call.input}
#     # print_tool_call(call.tool_name, args)
#     print(call.tool_name, args)

#     output_parts = [str(d) for d in call.output_deltas]
#     if output_parts:
#         # print_tool_result(call.tool_name, "".join(output_parts))
#         print(call.tool_name, "".join(output_parts))

#     if call.error:
#         print(f"[{call.tool_name}] {call.error}")

In [18]:
import json

In [19]:
def _parse_tool_call_chunks(message) -> None:
    """Accumulate tool_call_chunks by id, parse complete JSON args, print once."""
    pending: dict[str, dict] = {}
    for chunk in message.tool_calls:
        call_id = chunk.get("id") or str(chunk.get("index", 0))
        if call_id not in pending:
            pending[call_id] = {"name": "", "args_raw": ""}
        if chunk.get("name"):
            pending[call_id]["name"] = chunk["name"]
        if chunk.get("args") is not None:
            pending[call_id]["args_raw"] += chunk["args"]
 
    for call in pending.values():
        if not call["name"]:
            continue
        args: dict = {}
        raw = call["args_raw"].strip()
        if raw:
            try:
                args = json.loads(raw)
            except Exception:
                args = {"args": raw}
        print(call["name"], args)

In [22]:
from langchain.agents import create_agent
from langchain.messages import AIMessageChunk
from langchain_anthropic import ChatAnthropic
from langchain_core.runnables import Runnable

user_input = 'what files are in testing.ipynb'
agent_input = {"messages": [{"role": "user", "content": user_input}]}

agent = create_deep_agent(
    model=llm,
    backend=backend,
)

stream = agent.stream_events(agent_input, version="v3")

for message in stream.messages:

    for delta in message.reasoning:
        print(f"[thinking] {delta}", end="", flush=True)

    for delta in message.text:
        print(delta, end="", flush=True)

    for chunk in message.tool_calls:
        print(f"tool call chunk: {chunk}")

    finalized = message.tool_calls.get()
    if finalized:
        print(f"finalized tool calls: {finalized}")

for call in stream.tool_calls:
    print(f"{call.tool_name}({call.input})")
    for delta in call.output_deltas:
        print(delta, end="", flush=True)
    print(call.output, call.error)

    # _parse_tool_call_chunks(message)

tool call chunk: {'type': 'tool_call_chunk', 'id': '320683432', 'name': 'read_file', 'args': '', 'index': 0}
tool call chunk: {'type': 'tool_call_chunk', 'id': '320683432', 'name': 'read_file', 'args': '{"file_path":"/testing.ipynb"}', 'index': 0}
finalized tool calls: [{'type': 'tool_call', 'id': '320683432', 'name': 'read_file', 'args': {'file_path': '/testing.ipynb'}}]
The `testing.ipynb` file contains **4 code cells**:

1. **Cell 1** (empty) - No content
2. **Cell 2** (empty) - No content  
3. **Cell 3** (empty) - No content
4. **Cell 4** (execution_count: 1) - Contains a `ChatOpenAI` initialization for the Qwen model

The file appears to be mostly empty with only one cell containing actual code that initializes an LLM client.

In [ ]:
stream = agent.stream_events(agent_input, version="v3")

for message in stream.messages:
    for delta in message.reasoning:
        print(f"[thinking] {delta}", end="", flush=True)

    for delta in message.text:
        print(delta, end="", flush=True)





Files in the root folder:
- `.git/` (directory)
- `.gitignore`
- `.mira_sessions/` (directory)
- `mira/` (directory)
- `mira_prompt.md`
- `testing.ipynb`

In [ ]:
stream = agent.stream_events(agent_input, version="v3")

for message in stream.messages:
    print(f"[{message.node}] ", end="")
    for delta in message.text:
        print(delta, end="", flush=True)

    full_message = message.output
    usage = full_message.usage_metadata
    if usage:
        print(usage)

[model] 

[model] 